# 03 - Fixed-Capacity Interruption Analysis

**Purpose:** outline how this energy component model fits into the wider `mu-star` workflow: supplied fixed-capacity system tables plus disrupted assets in, disruption-to-supply metrics out.

**Before running:**

- run `00_data_intake.ipynb` and `01_operational_network.ipynb` first;
- use the repository `.venv` kernel;
- review or create `existing_generators.csv`, `existing_lines.csv`, and `demand_profile.csv` under `data/1-processed/energy/collaborator`;
- keep `service_weights.csv` aligned with the chosen demand-allocation method.

**Expected analysis inputs:**

- `snapped_substations.parquet`, with one row per bus/substation and a `bus_id` column;
- `service_weights.csv`, with `bus_id` and `service_weight` values covering all substations and summing to one when a system-wide demand profile is used;
- `existing_generators.csv`, with `generator_id`, `bus_id`, `carrier`, `capacity_mw`, and `marginal_cost` populated;
- `existing_lines.csv`, with `line_id`, `bus0`, `bus1`, `v_nom_kv`, `length_km`, and `s_nom_mva` populated;
- `demand_profile.csv`, with a timestamp column plus either one `demand_mw` column or one complete demand column per `bus_id`;
- optional damage or outage rows with `component`, `asset_id`, and either `damage_fraction` or `available_fraction`.

**Analysis scope:** check which interruption-model inputs are present, show the mu-star handoff from asset damage to disrupted assets, load the supplied system tables, build the fixed-capacity PyPSA network when all required files exist, and compare normal operation with selected outage cases.

**Current limitation:** the analysis remains conditional until the generator register, electrical line table and dated demand profile are complete. Record whether the supplied system represents the current network, a reviewed future network or a temporary scenario before interpreting the results.

In [ ]:
import geopandas as gpd
import pandas as pd

from mu_star_energy.damage import damage_to_disruptions
from mu_star_energy.model import EnergyModel
from mu_star_energy.network import build_operational_network
from mu_star_energy.paths import output_energy_dir, processed_energy_dir

COLLABORATOR_DIR = processed_energy_dir() / "collaborator"
RESULTS_DIR = output_energy_dir()

print("Reading processed inputs from:", COLLABORATOR_DIR)
print("Suggested output folder:", RESULTS_DIR)

## How this ties into mu-star

The upstream `nismod/mu-star` repository treats each infrastructure model as a separate package under `src/<model_name>` with a standard interface. For the energy model, the practical handoff is:

- reviewed system data describing assets and usage;
- a list of disrupted or non-functional assets;
- disruption metrics that can feed later indirect-loss analysis and the results viewer.

In this repository, the reviewed system data are the cleaned bus, line, generator and demand tables. The disruption list is a table with `component`, `asset_id`, and `available_fraction`. The helper `damage_to_disruptions(...)` converts asset damage into that interface when other parts of mu-star provide `damage_fraction`.

In [ ]:
asset_damage_example = pd.DataFrame(
    {
        "component": ["Line", "Generator", "Bus"],
        "asset_id": ["LINE_ID_HERE", "GENERATOR_ID_HERE", "BUS_ID_HERE"],
        "damage_fraction": [1.0, 0.25, 0.0],
    }
)

disruption_example = damage_to_disruptions(asset_damage_example)

display(asset_damage_example)
display(disruption_example)

## Current input status

The baseline interruption model needs reviewed substations, demand allocation weights, generators, electrical line connections and a dated demand profile. The generated power-station template is useful reference material, but it is not yet the final generator register.

In [ ]:
required_inputs = pd.DataFrame(
    [
        {
            "file_name": "snapped_substations.parquet",
            "role": "buses",
            "required_now": True,
            "notes": "Produced by the intake notebook and used as the model buses.",
        },
        {
            "file_name": "service_weights.csv",
            "role": "demand allocation",
            "required_now": True,
            "notes": "Needed when demand_profile.csv uses one Mauritius-wide demand_mw column.",
        },
        {
            "file_name": "existing_generators.csv",
            "role": "generation register",
            "required_now": True,
            "notes": "Reviewed generator data copied from the template and completed with bus, fuel, capacity and cost.",
        },
        {
            "file_name": "existing_lines.csv",
            "role": "electrical topology",
            "required_now": True,
            "notes": "Reviewed line or transformer connections with bus endpoints and power ratings.",
        },
        {
            "file_name": "demand_profile.csv",
            "role": "time-varying demand",
            "required_now": True,
            "notes": "First column must be timestamps; remaining columns are demand_mw or one column per bus_id.",
        },
        {
            "file_name": "generation_register_template.csv",
            "role": "reference only",
            "required_now": False,
            "notes": "Generated helper file to copy into existing_generators.csv before review.",
        },
    ]
)
required_inputs["present"] = required_inputs["file_name"].map(
    lambda name: (COLLABORATOR_DIR / name).exists()
)
display(required_inputs[["file_name", "role", "required_now", "present", "notes"]])

## Capacity and fuel-energy basis

- Existing power stations use `capacity_mw` as electrical output capacity (`Generator.p_nom`) in `MW_e`. It is not an LHV fuel-input capacity.
- Generator `marginal_cost` is per `MWh_e` produced. Convert an LHV fuel price using `fuel price / efficiency + variable operating cost`.
- AC lines and transformers use apparent-power ratings in MVA; voltage is the nominal bus voltage in kV. The current builder creates AC lines only; transformer-table support is still to be added.
- PyPSA `Link.p_nom` is input-side power at `bus0`. Conversion output at `bus1` is `p_nom * efficiency`, so output-rated conversion capacity must be divided by efficiency before it is entered as `p_nom`.

PyPSA does not select LHV or HHV automatically, so the source basis must be recorded. The asset model currently uses `Generator` for power stations and `Line` for passive AC transmission. Explicit fuel, hydrogen and ammonia conversion chains belong to `Link`-based models. See the repository README for the full convention.


## Load the reviewed tables

The network builder already enforces the core data rules: generators need `generator_id`, `bus_id`, `carrier`, `capacity_mw`, and `marginal_cost`; lines need `line_id`, `bus0`, `bus1`, `v_nom_kv`, `length_km`, and `s_nom_mva`; the demand profile must use a regular time step and cannot contain missing or negative demand.

In [ ]:
def read_optional_csv(path):
    return pd.read_csv(path) if path.exists() else None


def read_demand_profile(path):
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    timestamp_column = frame.columns[0]
    frame[timestamp_column] = pd.to_datetime(frame[timestamp_column])
    return frame.set_index(timestamp_column)


buses = gpd.read_parquet(COLLABORATOR_DIR / "snapped_substations.parquet")
service_weights = pd.read_csv(COLLABORATOR_DIR / "service_weights.csv")
generator_template = pd.read_csv(COLLABORATOR_DIR / "generation_register_template.csv")

existing_generators = read_optional_csv(COLLABORATOR_DIR / "existing_generators.csv")
existing_lines = read_optional_csv(COLLABORATOR_DIR / "existing_lines.csv")
demand_profile = read_demand_profile(COLLABORATOR_DIR / "demand_profile.csv")

loaded_tables = pd.Series(
    {
        "buses": len(buses),
        "service_weights": len(service_weights),
        "generation_register_template": len(generator_template),
        "existing_generators": len(existing_generators) if existing_generators is not None else "missing",
        "existing_lines": len(existing_lines) if existing_lines is not None else "missing",
        "demand_profile": len(demand_profile) if demand_profile is not None else "missing",
    },
    name="rows",
)
display(loaded_tables.to_frame())

if existing_generators is None:
    display(generator_template.head())

## Distribution-network extension with GridFinder

The baseline model remains a reviewed transmission-level model. At present, GridFinder and OSM routes only influence `service_weights.csv`: mapped line length is used as a proxy for how total demand is divided between transmission substations. This does not model distribution connectivity, voltage drop, feeder limits or downstream outages.

A useful GridFinder experiment should be kept as a separate, explicitly synthetic scenario:

1. **Prepare a candidate graph.** Reproject the GridFinder routes, split them at intersections, remove duplicate and very short segments, and retain `source=gridfinder` on every edge.
2. **Anchor feeder roots.** Connect each graph component to a reviewed transmission substation only where the snap distance is below a documented threshold. Leave unanchored components visible rather than forcing a connection.
3. **Place demand on the graph.** Allocate demand to buildings, population cells or night-light locations, then calibrate those loads so their total and substation shares match the selected demand profile and `service_weights.csv`.
4. **Reduce the graph.** Collapse degree-two geometry nodes while preserving feeder roots, branches, demand nodes and edge lengths. This keeps the PyPSA model tractable.
5. **Represent voltage levels explicitly.** Add separate transmission and distribution buses with transformers between them. The current network builder supports AC lines only, so transformer-table support is required before this step is electrically meaningful.
6. **Run assumption sets, not a single claimed network.** Voltage, conductor resistance/reactance, thermal capacity and normally-open points are not supplied by GridFinder. Store them in named low/base/high-capacity scenarios and report sensitivity of unserved energy to those assumptions.

The first defensible extension is a topology-only service model: a damaged synthetic feeder edge disconnects downstream demand, without claiming that inferred routes have known electrical capacities. Add distribution power flow only after the voltage, transformer and feeder-capacity assumptions are documented and validated against any CEB data that become available.

In [ ]:
distribution_proxy_status = pd.Series(
    {
        "substations represented": service_weights["bus_id"].nunique(),
        "GridFinder route length used (km)": service_weights.get(
            "gridfinder_km", pd.Series(dtype=float)
        ).sum(),
        "OSM route length used (km)": service_weights.get(
            "osm_km", pd.Series(dtype=float)
        ).sum(),
        "allocation methods": ", ".join(
            sorted(service_weights.get("method", pd.Series(dtype=str)).dropna().unique())
        ),
        "current electrical representation": "transmission buses only",
        "recommended next experiment": "topology-only synthetic feeders",
    },
    name="value",
)
display(distribution_proxy_status.to_frame())

## Build the fixed-capacity baseline network

This step should only use confirmed existing assets. The model rejects missing generator capacities, missing marginal costs, missing line ratings and irregular demand timestamps. It can divide one Mauritius-wide `demand_mw` series across substations using `service_weights.csv`, or it can read one demand column per `bus_id`.

In [ ]:
network = None
baseline_error = None
missing_for_network = []

if existing_generators is None:
    missing_for_network.append("existing_generators.csv")
if existing_lines is None:
    missing_for_network.append("existing_lines.csv")
if demand_profile is None:
    missing_for_network.append("demand_profile.csv")

if missing_for_network:
    print("Skipping network build until these files are available:")
    for file_name in missing_for_network:
        print(f"- {file_name}")
else:
    try:
        network = build_operational_network(
            buses=buses,
            lines=existing_lines,
            generators=existing_generators,
            demand_profile=demand_profile,
            service_weights=service_weights,
        )
        network_summary = pd.Series(
            {
                "buses": len(network.buses),
                "lines": len(network.lines),
                "generators": len(network.generators),
                "snapshots": len(network.snapshots),
                "time_step_hours": float(network.snapshot_weightings.generators.iloc[0]),
                "peak_demand_mw": float(network.loads_t.p_set.sum(axis=1).max()),
            },
            name="value",
        )
        display(network_summary.to_frame())
    except Exception as error:
        baseline_error = error
        print("The current reviewed inputs do not build yet:")
        print(error)

## Check normal operation before outages

Run one baseline case with no disruptions before adding damage or outages. In the final workflow this is the point to confirm that the reviewed network can meet demand without using the high-cost load-shedding generators, or to record any remaining unmet demand as a known data or modelling issue.

In [ ]:
baseline_result = None

if network is None:
    print("Skipping the normal-operation run because the baseline network was not built.")
elif baseline_error is not None:
    print("Skipping the normal-operation run because the network build reported an error.")
else:
    baseline_result = EnergyModel(solver_name="highs").simulate(network, [])
    display(pd.Series(baseline_result.metrics, name="baseline").to_frame())

    weights = baseline_result.network.snapshot_weightings.generators.reindex(
        baseline_result.network.snapshots
    ).fillna(1.0)
    shedding_columns = baseline_result.network.generators.index[
        baseline_result.network.generators.carrier.eq("load_shedding")
    ]
    baseline_shedding = baseline_result.network.generators_t.p.reindex(
        columns=shedding_columns
    ).clip(lower=0.0)
    baseline_shedding_by_bus = baseline_shedding.mul(weights, axis=0).sum(axis=0)
    baseline_shedding_by_bus.index = baseline_shedding_by_bus.index.str.replace(
        "load_shedding::", "", regex=False
    )
    display(baseline_shedding_by_bus.rename("unserved_energy_mwh").to_frame())

## Define outage cases

Each disruption row needs `component`, `asset_id`, and `available_fraction`. `component` can be `Generator`, `Line`, or `Bus`. A value of `0.0` means fully unavailable; `0.5` means half the normal capability remains. Start with a short reviewed case list before expanding to hazard-driven scenarios.

In [ ]:
example_generator = (
    str(existing_generators.loc[0, "generator_id"])
    if existing_generators is not None and not existing_generators.empty
    else "GENERATOR_ID_HERE"
)
example_line = (
    str(existing_lines.loc[0, "line_id"])
    if existing_lines is not None and not existing_lines.empty
    else "LINE_ID_HERE"
)
example_bus = str(buses.loc[0, "bus_id"]) if not buses.empty else "BUS_ID_HERE"

scenario_library = pd.DataFrame(
    [
        {
            "case": "single_generator_outage",
            "component": "Generator",
            "asset_id": example_generator,
            "available_fraction": 0.0,
            "reason": "Full outage of one reviewed generator.",
        },
        {
            "case": "line_derating",
            "component": "Line",
            "asset_id": example_line,
            "available_fraction": 0.5,
            "reason": "Partial loss of transfer capability after damage.",
        },
        {
            "case": "substation_outage",
            "component": "Bus",
            "asset_id": example_bus,
            "available_fraction": 0.0,
            "reason": "Full outage of one transmission substation.",
        },
    ]
)
display(scenario_library)

## Compare cases and write outputs

A completed run should save at least four things under `data/2-out/energy/`: the reviewed disruption table, summary metrics by case, unmet demand by substation, and any time-series tables needed for plotting. These are the local stand-ins for the disruption metrics that the wider mu-star workflow would later pass into indirect-loss analysis and the results viewer. Keep the normal-operation case in the same output structure so it is easy to compare with outages.

For mu-star tie-in, the most useful first outputs are whole-system metrics such as `served_fraction`, `unserved_energy_mwh`, and `objective`, plus bus-level unmet-demand tables that can be joined to demand zones, customer groups or later economic-impact calculations.

In [ ]:
def run_case(case_name, disruptions):
    result = EnergyModel(solver_name="highs").simulate(
        network, disruptions.to_dict(orient="records")
    )
    weights = result.network.snapshot_weightings.generators.reindex(
        result.network.snapshots
    ).fillna(1.0)
    shedding_columns = result.network.generators.index[
        result.network.generators.carrier.eq("load_shedding")
    ]
    shedding = result.network.generators_t.p.reindex(columns=shedding_columns).clip(
        lower=0.0
    )
    shedding_by_bus = shedding.mul(weights, axis=0).sum(axis=0)
    shedding_by_bus.index = shedding_by_bus.index.str.replace(
        "load_shedding::", "", regex=False
    )
    metrics = pd.Series({"case": case_name, **result.metrics})
    return metrics, shedding_by_bus.rename("unserved_energy_mwh")


case_metrics = None
case_shedding = None
placeholder_ids = scenario_library["asset_id"].str.endswith("_HERE").any()

if network is None:
    print("Scenario execution is the next step once the baseline network builds.")
elif placeholder_ids:
    print("Replace the placeholder asset IDs with reviewed IDs before running outage cases.")
else:
    metric_rows = [pd.Series({"case": "baseline", **baseline_result.metrics})]
    shedding_tables = [baseline_shedding_by_bus.rename("baseline")]
    for case_name, case_frame in scenario_library.groupby("case", sort=False):
        disruptions = case_frame[["component", "asset_id", "available_fraction"]]
        metrics, shedding_by_bus = run_case(case_name, disruptions)
        metric_rows.append(metrics)
        shedding_tables.append(shedding_by_bus.rename(case_name))

    case_metrics = pd.DataFrame(metric_rows)
    case_shedding = pd.concat(shedding_tables, axis=1).fillna(0.0)
    display(case_metrics)
    display(case_shedding)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    scenario_library.to_csv(RESULTS_DIR / "disruption_cases.csv", index=False)
    case_metrics.to_csv(RESULTS_DIR / "outage_case_metrics.csv", index=False)
    case_shedding.rename_axis("bus_id").to_csv(
        RESULTS_DIR / "unserved_energy_by_bus.csv"
    )
    print("Wrote interruption-analysis outputs to:", RESULTS_DIR)

## Modelling decisions to document

- Which reviewed model year should the generator, line and demand inputs represent?
- Is the first `demand_profile.csv` based on observed Mauritius demand, or is it a documented temporary profile derived from PyPSA-Earth or another source?
- Which outage cases are priority checks: single-asset failures, named historical events, or hazard-driven disruption tables?
- Which disruption metrics should move onward into mu-star indirect-loss and viewer steps first: whole-system service levels, bus-level unmet demand, customer-sector impacts, or operating-cost changes?
- For a synthetic GridFinder scenario, which feeder-root snap threshold, demand-location data and low/base/high electrical assumptions should be reviewed?